# EmotionCNN huấn luyện lại trên bộ CSV chung
Thí nghiệm mới: 22.295 train / 4.777 validation / 4.778 test, stratified 70/15/15 do người cung cấp mô tả. Giữ nguyên ba CSV; không lọc hoặc chia lại. B/C khởi tạo ngẫu nhiên, không nạp checkpoint thí nghiệm cũ. A là pretrained đối chứng, lịch sử dữ liệu train của A chưa được xác minh.

Trong Colab chọn GPU T4, lưu bản sao notebook vào Drive. Upload ba CSV vào `MyDrive/PBL4-FERPlus/common-csv-70-15-15-v1`. Model A hiện có đặt trong `MyDrive/PBL4-FERPlus/models/emotion-ferplus-8.onnx`. Chạy từng mục; phần test cuối mặc định tắt.


## 1 Lấy mã nguồn và cài môi trường

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil
BRANCH = 'thai+emotion-recognition-training'
REPO = Path('/content/PBL4')
if not REPO.exists():
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,'https://github.com/Thainguyen2103/Panda-Robotics-Client-.git',str(REPO)],check=True)
os.chdir(REPO)
subprocess.run(['git','pull','--ff-only'],check=True)
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip() == BRANCH
subprocess.run([sys.executable,'-m','pip','install','-q','-r','training/emotion/requirements.txt'],check=True)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
def run(module,*args):
    subprocess.run([sys.executable,'-u','-m',module,*map(str,args)],cwd=REPO,check=True)
print('Git:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


## 2 Kết nối Drive và cấu hình mới
Giữ nguyên cấu hình khi tiếp tục phiên. Nếu đổi cấu hình, đặt EXPERIMENT mới. Chạy seed 42 trước; thêm seed 123 và 2026 trước khi mở test nếu đủ thời gian.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Cần GPU để chạy train đầy đủ'
ROOT = Path('/content/drive/MyDrive/PBL4-FERPlus')
SOURCE = ROOT/'common-csv-70-15-15-v1'
EXPERIMENT = 'common701515-cnn-v1'
SEEDS = [42]
EPOCHS, BATCH, LR, WD, PATIENCE = 80, 128, 0.001, 0.0001, 12
RUNS = ROOT/'runs'/EXPERIMENT
EXPORTS = ROOT/'exports'/EXPERIMENT
REPORTS = ROOT/'reports'/EXPERIMENT
BASELINE = ROOT/'models/emotion-ferplus-8.onnx'
print(torch.cuda.get_device_name(0), SOURCE)


## 3 Kiểm tra CSV và chuẩn bị dữ liệu
Dữ liệu train được đọc từ máy Colab, checkpoint lưu Drive. CSV khác mã băm sẽ bị từ chối. Không tiếp tục nếu kiểm tra ảnh gần trùng phát hiện cặp xuyên tập.

In [ ]:
from training.emotion.common import sha256
from training.emotion.common_csv import prepare_csv
from training.emotion.data import load_split
lock = json.loads((REPO/'training/emotion/common_csv_lock.json').read_text())
assert {name:sha256(SOURCE/name) for name in lock['sources']} == lock['sources'], 'CSV khác bản đã chốt'
DATA = Path('/content/ferplus-common701515')
if not DATA.exists(): prepare_csv(SOURCE, DATA)
manifest = json.loads((DATA/'manifest.json').read_text())
assert manifest['sources'] == lock['sources']
assert not manifest['near_duplicate_check']['cross_split_pairs']
for split in ('train','validation','test'): load_split(DATA,split)
print('Fingerprint:',manifest['fingerprint'])
print(json.dumps(manifest['splits'],indent=2))


## 4 Huấn luyện B và C từ đầu
Tên thư mục mới tách khỏi checkpoint cũ. Resume chỉ dùng last.pt của đúng bộ dữ liệu và cấu hình này. Mỗi epoch in tiến độ ngay trong ô kết quả; không chạy ô dưới khi ô này chưa hoàn thành.

In [ ]:
RUN_DIRS = {}
for seed in SEEDS:
    for mode in ('hard','soft'):
        name = f'{mode}-seed{seed}'
        dest = RUNS/name
        args = ['--data',DATA,'--output',dest,'--mode',mode,'--seed',seed,
                '--epochs',EPOCHS,'--batch-size',BATCH,'--lr',LR,
                '--weight-decay',WD,'--patience',PATIENCE,'--device','cuda']
        if (dest/'last.pt').exists(): args += ['--resume']
        run('training.emotion.train',*args)
        RUN_DIRS[name] = dest
print('Hoàn thành:',list(RUN_DIRS))


## 5 Đồ thị huấn luyện và checkpoint

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2,figsize=(12,4))
for name,dest in RUN_DIRS.items():
    h=json.loads((dest/'history.json').read_text()); x=[r['epoch'] for r in h]
    axes[0].plot(x,[r['train']['loss'] for r in h],label=name+' train')
    axes[0].plot(x,[r['validation']['loss'] for r in h],'--',label=name+' val')
    axes[1].plot(x,[r['validation']['macro_f1'] for r in h],label=name)
for ax in axes: ax.set_xlabel('Epoch'); ax.legend()
axes[0].set_title('Loss'); axes[1].set_title('Validation macro F1')
plt.tight_layout();plt.show()


## 6 Xuất ONNX và kiểm tra tương đương
Chỉ xuất từ best.pt. Kiểm tra số học bằng PyTorch, ONNX Runtime và OpenCV trên validation và mẫu pixel biên.

In [ ]:
from training.emotion.export import export
MODELS = [('pretrained',BASELINE)]
for name,dest in RUN_DIRS.items():
    path=EXPORTS/f'{name}.onnx'
    if not path.exists(): export(dest/'best.pt',DATA,path)
    else:
        meta=json.loads(path.with_suffix('.onnx.json').read_text())
        assert meta['checkpoint_sha256']==sha256(dest/'best.pt')
        assert meta['dataset_fingerprint']==manifest['fingerprint']
        assert meta['model_sha256']==sha256(path)
    MODELS.append((name,path))


## 7 Đánh giá validation và chốt cấu hình
A dùng resize Lanczos theo model card; B/C giữ INTER_AREA như lúc học. Tất cả cùng ảnh và nhãn. Không chuẩn hóa ngoài model lần nữa. Thư mục báo cáo phải mới; nếu đã đo thì đọc kết quả đã lưu.

In [ ]:
from training.emotion.evaluate import compare
EXPECTED_A='a2a2ba6a335a3b29c21acb6272f962bd3d47f84952aaffa03b60986e04efa61c'
assert BASELINE.is_file() and sha256(BASELINE)==EXPECTED_A, 'Cần đúng file A đối chứng'
VAL_REPORT=REPORTS/'validation'
if not VAL_REPORT.exists(): compare(DATA,MODELS,VAL_REPORT,split='validation')
print((VAL_REPORT/'summary.csv').read_text())


## 8 Test cuối và xuất kết quả để đối chiếu
Chỉ bật sau khi chốt kiến trúc, seed, augmentation và checkpoint theo validation. Không dùng test để tinh chỉnh rồi gọi kết quả tiếp theo là đánh giá độc lập. predictions.csv chứa image_id, nhãn tham chiếu, nhãn dự đoán và tám điểm xác suất; trao đổi file này với bạn cùng nhóm, không cần gửi model. Tốc độ chỉ so khi cùng máy và điều kiện. A có lịch sử train không rõ nên không bảo đảm test mới chưa từng được A học.

In [ ]:
RUN_FINAL_TEST = False
TEST_REPORT=REPORTS/'final-test'
if RUN_FINAL_TEST:
    if not TEST_REPORT.exists(): compare(DATA,MODELS,TEST_REPORT,split='test')
    print((TEST_REPORT/'summary.csv').read_text())
    print('File trao đổi:',TEST_REPORT/'predictions.csv')
else:
    print('Test chưa mở. Chỉ dùng validation để lựa chọn.')


## 9 Kết quả cần lưu
Giữ manifest, mã băm CSV, config.json, history.json, status.json, best.pt, last.pt, ONNX và metadata, summary.csv và predictions.csv. Đồ thị và báo cáo mới chỉ được lập sau khi có kết quả train thực; notebook chưa chứa kết quả đầy đủ. Kết quả official Usage cũ được giữ làm thí nghiệm trước, không ghép vào bảng mới.

Nếu Colab mất phiên: kết nối GPU, chạy lại mục 1–3, rồi mục 4 để tiếp tục từ last.pt của thí nghiệm này. Dữ liệu đọc lại từ ba CSV gốc cố định. Không thay EPOCHS/LR khi resume.